In [ ]:
### IMPORTS ###
import os
import subprocess
from typing import Dict, Any, Callable

from dotenv import load_dotenv
import json
import requests

In [ ]:
### RETRIEVE TOKEN ###
load_dotenv('./src/env', override=True)

CLIENT_ID = os.getenv('CLIENT_ID')
CLIENT_SECRET = os.getenv('CLIENT_SECRET')

def get_token(client_id: str, client_secret: str, url: str) -> Dict[Any, Any]:
""" Allows to perform a POST request to retrieve an access token.

Args:
  client_id (str): App client id
  client_secret (str): App client secret
  url (str): URL to perform the post request

Returns:
  Dict[Any, Any]: Dictionary containing the access token
"""

headers = {
    'Content-Type': 'application/x-www-form-urlencoded'
}

payload = {
    'grant_type': 'client_credentials',
    'client_id': client_id,
    'client_secret': client_secret
}

try:
  response = requests.post(url=url, headers=headers, data=payload)
  print(type(response))
  response.raise_for_status()
  response_json = json.loads(response.content)

  return response_json

except Exception as err:
  print(f"Error: {err}")
  return {}

URL_TOKEN="https://accounts.spotify.com/api/token"
token = get_token(client_id=CLIENT_ID, client_secret=CLIENT_SECRET, url=URL_TOKEN)

print(token)

def get_auth_header(access_token: str) -> Dict[str, str]:
  return {"Authorization": f"Bearer {access_token}"}

In [ ]:
### GET FEATURED PLAYLISTS ###

def get_featured_playlists(url: str, access_token: str, offset: int=0, limit int=20, next: str="") -> Dict[Any, Any]:
  """Peform get() request to featured playlists endpoint

  Args:
    url (str): Base URL to perform the get request
    access_token (str): Access token to perform the request
    offset (int, optional): Offset to perform the request. Defaults to 0.
    limit (int, optional): Number of elements per page. Defaults to 20.
    next (str, optional): Next URL to perform next request. Defaults to "".

  Returns:
    Dict[Any, Any]: Reuqest response
  """

  if next == "":
    request_url = f"{url}?offset={offset}&limit={limit}"
  else:
    request_url = f"{next}"

  headers = get_auth_header(access_token=access_token)

  try:
    response = requests.get(url=request_url, headers=headers)
    return response.json

  except Exception as err:
    print(f"Error requesting data: {err}")
    return {"error": err}

URL_FEATURED_PLAYLISTS = "https://api.spotify.com/v1/browse/featured-playlists"

playlists_response = get_featured_playlists(url=URL_FEATURED_PLAYLISTS, access_token=token.get('access_token'))
playlists_response

# Structure of the response
playlists_response.keys()

playlists_response.get("playlists").keys()

playlists_response.get("playlists").get("href")

# Number of items returned; list item
len(playlists_resposne.get("playlists").get("items"))
playlists_response.get("playlists").get("items")[0]

# Next call
next_playlists_response = get_featured_playlists(url=URL_FEATURED_PLAYLISTS, access_token=token.get('access_token'), offset=20, limit=20)
next_playlists_response